## Table VII: Trust and Ledger Ablation (re-run of the CRTAT branch)

Re-runs the CRTAT branch with the same seeds, splits, participation stream and training RNG as the main run (early stopping off), so the 'trust + hash ledger' row reproduces Table V's CRTAT accuracy exactly (asserted in the printed check). Configuration (seeds, partition, stopping rule, averaging) is printed above the table, and runtime is split by scope: local training / detector / aggregation / ledger construction / ledger verification / total.


In [ ]:
# ─── Table VII: trust & ledger ablation — re-runs the CRTAT branch only ──────────────────────
# Uses the SAME code path (_client_update), the SAME participation stream default_rng(seed) and the
# SAME training stream method_rng(seed,'CRTAT+Blockchain') as run_fl_simulation, plus the same split
# split_dirichlet(alpha=0.5, seed). With early stopping OFF, variant 'CRTAT + trust + hash ledger'
# therefore reproduces the main-run CRTAT accuracy exactly (asserted below).
def run_crtat_variant(factory, client_data, X_test, y_test, input_dim, n_classes, seed,
                      decay=CRTAT_DECAY, use_ledger=False, aggregator='crtat', attackers=True,
                      boost=1.0, clip=False, missing='neutral'):
    t_run0 = time.perf_counter()
    tm = dict(train_s=0.0, detect_s=0.0, aggregate_s=0.0, ledger_build_s=0.0, ledger_verify_s=0.0)
    rng_s = np.random.default_rng(seed)                                   # participation stream
    rng_t = method_rng(seed, 'CRTAT+Blockchain' if aggregator == 'crtat' else 'FedAvg_NoDefense')
    gp    = factory()[0].get_params().copy()
    crtat = CRTAT(decay=decay, missing=missing)
    bc    = TrustBlockchain() if use_ledger else None
    n_select = max(1, int(round(NUM_CLIENTS * CLIENT_FRACTION)))
    T, M, S = [], [], []
    for rnd in range(1, FL_ROUNDS + 1):
        lr = cosine_lr(rnd, FL_ROUNDS)
        participating = sorted(rng_s.choice(NUM_CLIENTS, n_select, replace=False).tolist())
        t0 = time.perf_counter(); clients, updates = factory(), []
        for cid in participating:
            X_c, y_c = client_data[cid]
            updates.append(_client_update(clients[cid], gp, X_c, y_c, cid, rnd, FL_ROUNDS, lr, rng_t,
                                          boost=boost, attackers=attackers))
        tm['train_s'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        trust_full = np.full(NUM_CLIENTS, NEUTRAL_TRUST); inv_full = np.zeros(NUM_CLIENTS)
        trust_full[participating] = blockchain_trust(updates)
        inv_full[participating]   = inversion_signal(updates)
        mask = np.zeros(NUM_CLIENTS, dtype=bool); mask[participating] = True
        crtat.update(trust_full, participated_mask=mask, inv_scores=inv_full)
        sc = crtat.combined_score(trust_full)
        tm['detect_s'] += time.perf_counter() - t0

        if bc is not None:
            t0 = time.perf_counter(); bc.add_round(rnd, trust_full, sc, [], FLAG_THRESHOLD)
            tm['ledger_build_s'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        agg_in = clip_to_median_norm(updates) if clip else updates
        w = crtat_weights(sc[participating]) if aggregator == 'crtat' else None
        gp = gp + fedavg_agg(agg_in, w)
        tm['aggregate_s'] += time.perf_counter() - t0
        T.append(trust_full); M.append(mask); S.append(np.where(mask, sc, np.nan))

    T, M, S = np.array(T), np.array(M), np.array(S)
    det = final_signal(S)
    hon = np.zeros(NUM_CLIENTS, bool); hon[HONEST_CLIENTS] = True; mal = ~hon
    cA = confusion_counts(gt_labels, OPERATING_POINTS['OP-A']['fn'](det))
    cB = confusion_counts(gt_labels, OPERATING_POINTS['OP-B']['fn'](det))
    out = dict(acc=evaluate_model(gp, X_test, y_test, input_dim=input_dim, n_classes=n_classes)[0]*100,
               attack=attack_metrics(gp, X_test, y_test, input_dim, n_classes), params=gp,
               auc=auc_pooled(det), f1_A=cA['f1'], f1_B=cB['f1'], nflag_A=cA['n_flagged'], conf_A=cA, conf_B=cB,
               trust_h=T[M & hon[None, :]].mean(), trust_m=T[M & mal[None, :]].mean(), ledger='—')
    if bc is not None:
        t0 = time.perf_counter(); ok, _ = bc.is_valid(); tm['ledger_verify_s'] = time.perf_counter() - t0
        out['ledger'] = (f'{len(bc.chain)-1} blocks, '
                         f'{len(_json.dumps([b.__dict__ for b in bc.chain], default=str))/1024:.1f} KB, '
                         f'{"valid" if ok else "INVALID"}')
    tm['total_s'] = time.perf_counter() - t_run0
    out['timing'] = tm
    return out

TABLE_VII_VARIANTS = [
    ('CRTAT without trust recovery', dict(decay=1.0,         use_ledger=False)),
    ('CRTAT + trust',                dict(decay=CRTAT_DECAY, use_ledger=False)),
    ('CRTAT + trust + hash ledger',  dict(decay=CRTAT_DECAY, use_ledger=True)),
]

def table_vii(dname, factory, X, y, X_test, y_test, input_dim, main_results, n_classes=N_CLASSES):
    splits = {s: split_dirichlet(X, y, NUM_CLIENTS, alpha=0.5, seed=s) for s in SEEDS}
    print(f'\nTable VII — Trust and blockchain ablation | {dname}')
    print(f'  Configuration: seeds={SEEDS} | partition=Dirichlet(alpha=0.5, seed=s) | {NUM_CLIENTS} clients, '
          f'{int(round(NUM_CLIENTS*CLIENT_FRACTION))} participate/round | rounds={FL_ROUNDS}')
    print(f'  Stopping rule: none (all {FL_ROUNDS} rounds; early stopping OFF) | accuracy = test accuracy after the last round')
    print(f'  Averaging: mean ± std over seeds (population std) | Honest/Malicious Trust = mean cosine-trust over participating rounds')
    print(f'  Timing columns (seconds, per full {FL_ROUNDS}-round run; mean over seeds): '
          f'train = local training of participants | detect = trust+CRTAT scoring | agg = weighted aggregation | '
          f'ledger-build = block construction | ledger-verify = one is_valid() pass | total = whole run')
    print('═'*172)
    print(f'{"Variant":<30}{"Final Acc %":>16}{"F1 OP-A":>10}{"F1 OP-B":>10}{"#flag A":>8}{"Hon.Trust":>10}{"Mal.Trust":>10}'
          f'{"train":>8}{"detect":>8}{"agg":>8}{"l-build":>9}{"l-verify":>10}{"total":>8}  Ledger')
    print('─'*172)
    allr = {}
    for name, kw in TABLE_VII_VARIANTS:
        rs = [run_crtat_variant(factory, splits[s], X_test, y_test, input_dim, n_classes, s, **kw) for s in SEEDS]
        allr[name] = rs
        a = np.array([r['acc'] for r in rs]); g = lambda k: np.mean([r[k] for r in rs])
        t = lambda k: np.mean([r['timing'][k] for r in rs])
        print(f'{name:<30}{f"{a.mean():.2f}±{a.std():.2f}":>16}{g("f1_A"):>10.3f}{g("f1_B"):>10.3f}{g("nflag_A"):>8.1f}'
              f'{g("trust_h"):>10.3f}{g("trust_m"):>10.3f}{t("train_s"):>8.2f}{t("detect_s"):>8.3f}{t("aggregate_s"):>8.3f}'
              f'{t("ledger_build_s"):>9.4f}{t("ledger_verify_s"):>10.4f}{t("total_s"):>8.2f}  {rs[0]["ledger"]}')
    print('═'*172)
    # reproducibility: this variant must equal the CRTAT branch of the main run (Table V) seed by seed
    if all(r[11]['early_stop'] is False for r in main_results):
        diffs = [abs(allr['CRTAT + trust + hash ledger'][i]['acc'] - main_results[i][0]['CRTAT+Blockchain']['acc'][-1]*100)
                 for i in range(len(SEEDS))]
        ok = max(diffs) < 1e-6
        print(f'  Reproduction check vs main run (Table V CRTAT accuracy, per seed): max|Δ|={max(diffs):.2e} pp → '
              f'{"REPRODUCED ✅" if ok else "DIFFERS ❌ — configurations are not identical; investigate before reporting"}')
    else:
        print('  Reproduction check skipped: main run used early stopping, so round counts differ.')
    return allr

tab7_cifar = table_vii('CIFAR-10', clients_factory_cifar, Xf_cifar, y_cifar, Xf_cifar_test, y_cifar_test, CIFAR_PCA_DIM, results_cifar)
tab7_mnist = table_vii('MNIST',    clients_factory_mnist, Xf_mnist, y_mnist, Xf_mnist_test, y_mnist_test, MNIST_PCA_DIM, results_mnist)



Table VII — Trust and blockchain ablation | CIFAR-10
  Configuration: seeds=[42, 123] | partition=Dirichlet(alpha=0.5, seed=s) | 30 clients, 12 participate/round | rounds=15
  Stopping rule: none (all 15 rounds; early stopping OFF) | accuracy = test accuracy after the last round
  Averaging: mean ± std over seeds (population std) | Honest/Malicious Trust = mean cosine-trust over participating rounds
  Timing columns (seconds, per full 15-round run; mean over seeds): train = local training of participants | detect = trust+CRTAT scoring | agg = weighted aggregation | ledger-build = block construction | ledger-verify = one is_valid() pass | total = whole run
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Variant                            Final Acc %   F1 OP-A   F1 OP-B #flag A Hon.Trust Mal.Trust   train  detect     agg  l-build  l-verify   total  Ledger
────────